# TD 8 — Corrigé

**Analyse des données — L3 Économie**

Mis en ligne après la séance.

> Les DVF sont réactualisées deux fois par an : exécutez le notebook pour obtenir les chiffres du jour. Les ordres de grandeur et le **sens** des déplacements, eux, ne changent pas.


## Partie 1 — Ce que décrit une ligne

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

BASE = "https://files.data.gouv.fr/geo-dvf/latest/csv/{annee}/departements/{dep}.csv.gz"
ANNEE = 2024
DEPTS = ["95", "78", "60"]          # Val-d'Oise, Yvelines, Oise

# D'abord : que contient ce fichier ? On lit cinq lignes avant de tout charger.
apercu = pd.read_csv(BASE.format(annee=ANNEE, dep="95"), nrows=5)
print(len(apercu.columns), "colonnes\n")
print(apercu.columns.tolist())

In [ ]:
COLS = ["id_mutation", "date_mutation", "nature_mutation", "valeur_fonciere",
        "code_commune", "nom_commune", "code_departement",
        "type_local", "surface_reelle_bati", "nombre_pieces_principales",
        "surface_terrain"]

morceaux = []
for dep in DEPTS:
    m = pd.read_csv(BASE.format(annee=ANNEE, dep=dep), usecols=COLS,
                    dtype={"code_commune": str, "code_departement": str})
    print(dep, ":", len(m), "lignes")
    morceaux.append(m)

df = pd.concat(morceaux, ignore_index=True)
print("\ntotal :", len(df), "lignes |", df["id_mutation"].nunique(), "mutations")
df.head(3)

In [ ]:
gros = df["id_mutation"].value_counts()
print("mutations a plus d'une ligne :", (gros > 1).sum(),
      "soit", round(100 * (gros > 1).mean(), 1), "%")
print("maximum de lignes pour une mutation :", gros.max())
print()
exemple = gros.index[0]
print(df[df["id_mutation"] == exemple][["valeur_fonciere", "type_local",
                                        "surface_reelle_bati", "surface_terrain"]]
        .to_string(index=False))

**Réponse 1 et 2.** Le fichier compte nettement plus de **lignes** que de **mutations**. Une ligne n'est donc pas une vente.

Une ligne décrit un **local ou une parcelle** vendu dans une mutation. Vendre une maison avec son garage et son jardin produit trois lignes : la maison, la dépendance, le terrain. Et sur chacune, `valeur_fonciere` porte **le prix total de la vente**, répété à l'identique — ce n'est pas le prix du garage.

C'est exactement la situation des 2,3 millions de lignes DVF rencontrée au contrôle continu, et la raison pour laquelle la première question devant un fichier est toujours : *que décrit une ligne ?*


In [ ]:
naive  = df["valeur_fonciere"].sum()
correct = df.drop_duplicates("id_mutation")["valeur_fonciere"].sum()

print("somme naive   :", round(naive / 1e9, 2), "milliards d'euros")
print("somme correcte:", round(correct / 1e9, 2), "milliards d'euros")
print("surestimation :", round(100 * (naive / correct - 1), 1), "%")

**Réponse 3.** La somme naïve compte plusieurs fois le même prix : une fois par ligne de la mutation. Elle **surestime** le marché, et pas de quelques pour cent.

Un article fondé sur ce chiffre ne serait pas « imprécis » : il serait **faux**, et d'une erreur qu'aucun arrondi n'explique. Le dédoublonnage sur `id_mutation` est ici l'opération qui sépare un chiffre publiable d'un chiffre inventé.


In [ ]:
# une ligne = un logement vendu seul
lgt = df[df["type_local"].isin(["Appartement", "Maison"])].copy()
print("lignes de logement        :", len(lgt))

taille = lgt.groupby("id_mutation")["id_mutation"].transform("size")
d = lgt[taille == 1].copy()
print("mutations a un seul local :", len(d))

d = d[d["nature_mutation"] == "Vente"]
print("ventes uniquement         :", len(d))

d = d.dropna(subset=["valeur_fonciere", "surface_reelle_bati"])
d = d[d["surface_reelle_bati"] >= 9]
print("surface renseignee >= 9 m2:", len(d))

**Réponse 4.** Quatre filtres, dans cet ordre, et chacun se justifie.

**Les logements.** On écarte les dépendances, les terrains nus et les locaux industriels : ils n'ont pas de prix au mètre carré comparable.

**Les mutations à un seul local.** C'est le filtre décisif. Si une mutation contient deux appartements, le prix total ne se rattache à aucun des deux, et le diviser par la surface de l'un n'a aucun sens. On ne garde donc que les ventes dont le prix est **attribuable** à un bien unique. C'est aussi ce qui élimine les ventes en bloc.

**Les ventes.** `nature_mutation` contient aussi des adjudications, des échanges et des expropriations, dont les prix ne se forment pas sur le marché.

**La surface.** Sans elle, pas de prix au mètre carré. Le seuil de 9 m² écarte les erreurs de saisie — c'est la surface minimale d'un logement décent en droit français.


## Partie 2 — La régression hédonique

In [ ]:
d["prix_m2"] = d["valeur_fonciere"] / d["surface_reelle_bati"]
print("avant bornes : min", round(d["prix_m2"].min(), 2),
      "| median", round(d["prix_m2"].median()),
      "| max", round(d["prix_m2"].max()))

d = d[(d["prix_m2"] > 300) & (d["prix_m2"] < 20000)].copy()
print("apres bornes :", len(d), "logements |", d["code_commune"].nunique(), "communes")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(d["prix_m2"], bins=60, color="#9AA0A0", edgecolor="white", linewidth=.3)
ax.axvline(d["prix_m2"].median(), color="#E2162F", lw=1.6)
ax.set_title("Distribution du prix au mètre carré")
ax.set_xlabel("Prix au mètre carré (euros)")
ax.set_ylabel("Nombre de ventes")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.02, -0.03, "Source : DVF, ventes de logements en 95, 78 et 60.",
         fontsize=8, color="#7A8280")
plt.show()

**Réponse 5.** Avant bornes, le minimum et le maximum sont absurdes.

Une vente à **1 €** n'est pas une erreur de saisie : c'est une transaction réelle dont le prix n'est pas un prix de marché — donation déguisée, cession entre sociétés d'un même groupe, vente à une collectivité. Le champ est juste, mais l'observation ne relève pas de la question posée. C'est le même raisonnement que pour les salaires nuls de la séance 3 : on ne « corrige » pas, on **sort du champ** et on le dit.

Les bornes 300 € et 20 000 € sont un choix défendable, pas une vérité. Le bon réflexe est de dire combien d'observations elles écartent, et de vérifier que les coefficients n'en dépendent pas.


In [ ]:
d["lprix"]  = np.log(d["valeur_fonciere"])
d["lsurf"]  = np.log(d["surface_reelle_bati"])
d["maison"] = (d["type_local"] == "Maison").astype(int)
d["pieces"] = d["nombre_pieces_principales"]
d = d.dropna(subset=["lprix", "lsurf", "pieces"])
print(len(d), "observations pour l'estimation")

In [ ]:
m1 = smf.ols("lprix ~ lsurf", data=d).fit(cov_type="HC1")
print(m1.summary().tables[1])
print("\nR2 =", round(m1.rsquared, 4))

**Réponse 6.** Les deux variables étant en logarithme, le coefficient est une **élasticité** :

> Un logement **1 % plus grand** se vend, en moyenne, *b* **% plus cher**.

Pourquoi pas 1 ? Parce qu'une élasticité de 1 signifierait que le prix au mètre carré est le même quelle que soit la taille — qu'un studio et un cinq-pièces valent autant au m². Ce n'est pas le cas : le **prix au mètre carré décroît avec la surface**. Les petites surfaces sont plus demandées par mètre carré, et une partie de la valeur d'un logement ne dépend pas de sa taille — l'adresse, l'accès aux transports, le fait même d'avoir un toit.

Une élasticité inférieure à 1 est donc le résultat attendu, et il se lit directement : doubler la surface augmente le prix de moins du double.


In [ ]:
m2 = smf.ols("lprix ~ lsurf + pieces + maison", data=d).fit(cov_type="HC1")
print(m2.summary().tables[1])
print("\nlsurf :", round(m1.params["lsurf"], 4), "->", round(m2.params["lsurf"], 4))

**Réponse 7.** Le coefficient de la surface **baisse**, et c'est mécanique : surface et nombre de pièces sont fortement corrélés. Seul, `lsurf` portait les deux effets ; à nombre de pièces donné, il ne porte plus que le sien.

Le coefficient des pièces se lit désormais **à surface constante** — ce qui en change complètement le sens. Il ne compare pas un deux-pièces à un quatre-pièces, mais deux logements de **même surface** dont l'un est découpé en plus de pièces. Une pièce de plus à surface égale, ce sont des pièces plus petites. C'est pourquoi ce coefficient est faible, et pourquoi sa valeur n'a rien d'intuitif tant qu'on n'a pas dit ce qui est tenu constant.


## Partie 3 — Les effets fixes

In [ ]:
SPECS = {
 "M1 surface seule"     : "lprix ~ lsurf",
 "M2 + pieces + maison" : "lprix ~ lsurf + pieces + maison",
 "M3 + EF departement"  : "lprix ~ lsurf + pieces + maison + C(code_departement)",
 "M4 + EF commune"      : "lprix ~ lsurf + pieces + maison + C(code_commune)",
}

res = {}
for nom, f in SPECS.items():
    res[nom] = smf.ols(f, data=d).fit(cov_type="HC1")

def ligne(m):
    out = {v: round(m.params[v], 4) if v in m.params else None
           for v in ["lsurf", "pieces", "maison"]}
    out["R2"] = round(m.rsquared, 4)
    out["parametres"] = len(m.params)
    return out

tab = pd.DataFrame({nom: ligne(m) for nom, m in res.items()})
print(tab.to_string())

**Réponse 8.** `C(code_departement)` crée **deux** muettes pour trois départements : le troisième est la référence, logée dans la constante. C'est la règle des $k-1$ muettes de la séance 7.

Ce qu'elles absorbent : **tout ce qui est commun à un département et qui ne varie pas en son sein**. Le niveau général des prix, la distance à Paris, le marché du travail local, la fiscalité départementale. Trois chiffres suffisent à capter une part considérable de la variance, parce que la géographie est le premier déterminant du prix immobilier.

**Réponse 9.** Avec `C(code_commune)`, le modèle passe à plus d'une centaine de paramètres — une muette par commune moins une. Le $R^2$ monte encore nettement.

Des trois coefficients, c'est **`maison`** qui se déplace le plus. Celui de la surface, lui, bouge à peine : la relation entre taille et prix est à peu près la même partout, et les effets fixes ne la perturbent donc pas. Un coefficient stable quand on ajoute des effets fixes est un bon signe — il indique que la variable n'est pas une procuration de la géographie.


In [ ]:
par_commune = d.groupby("code_commune").agg(part_maison=("maison", "mean"),
                                            prix_m2=("prix_m2", "mean"),
                                            ventes=("maison", "size"))
par_commune = par_commune[par_commune["ventes"] >= 30]
print(len(par_commune), "communes avec au moins 30 ventes")
print("correlation part de maisons / prix au m2 :",
      round(par_commune["part_maison"].corr(par_commune["prix_m2"]), 3))

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.scatter(par_commune["part_maison"], par_commune["prix_m2"], s=20, alpha=.6,
           color="#3D4543")
ax.set_xlabel("Part de maisons dans les ventes de la commune")
ax.set_ylabel("Prix moyen au mètre carré (euros)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

**Réponse 10.** Le mécanisme tient en deux phrases.

Les maisons sont **concentrées dans les communes les moins chères** : les centres denses et bien desservis sont faits d'appartements, le périurbain de maisons. Sans effet fixe, la muette `maison` ne compare donc pas une maison à un appartement *du même endroit* — elle compare le périurbain au centre-ville, et le coefficient mesure surtout cette différence de localisation.

Les effets fixes communaux forcent la comparaison **à l'intérieur de chaque commune**. Le coefficient de `maison` répond alors à la question qu'on voulait poser : à surface, pièces et commune identiques, une maison vaut-elle plus ou moins qu'un appartement ?

C'est précisément cela, le *déplacement des coefficients* : il ne corrige pas une erreur de calcul, il change la **question à laquelle le coefficient répond**.


**Réponse 11.** Avec des effets fixes communaux, **toute variable constante à l'intérieur d'une commune disparaît** : elle est parfaitement colinéaire avec les muettes, et `statsmodels` l'écarte ou renvoie une erreur.

Deux exemples qui intéresseraient un économiste urbain :

- la **distance au centre de Paris**, ou le temps de trajet en RER : une seule valeur par commune ;
- le **revenu médian communal**, le taux de taxe foncière, la présence d'une gare, le classement du collège de secteur.

Autrement dit, on ne peut plus estimer l'effet de ce qui *fait* la commune. C'est le prix à payer, et il est lourd : **un effet fixe ne corrige rien, il absorbe**. Il absorbe le biais que l'on craignait, mais aussi la variable que l'on voulait mesurer. Choisir des effets fixes communaux, c'est renoncer à expliquer pourquoi une commune est chère, pour mieux estimer ce qui fait varier les prix *en son sein*.

**Réponse 12.** Ce qu'ils n'absorbent pas : tout ce qui varie **à l'intérieur** d'une commune. L'étage et l'ascenseur, l'état du bien et les travaux à prévoir, la rue et la vue, l'année de construction, la performance énergétique, la proximité immédiate d'une gare ou d'une nuisance. Aucune de ces variables n'est dans les DVF — c'est leur limite principale.

Le modèle à publier est **M4**, avec les effets fixes communaux, en affichant M1 à M3 à côté pour montrer le déplacement : c'est le tableau que porte tout article d'économie urbaine. Et son coefficient de surface ne mesure **pas** l'effet causal d'un mètre carré supplémentaire. Les grands logements diffèrent des petits par bien autre chose que leur taille — standing, étage, état — et le coefficient absorbe tout ce qui est corrélé aux deux à la fois.


## Pour ceux qui ont terminé

In [ ]:
m4 = res["M4 + EF commune"]
m4c = smf.ols(SPECS["M4 + EF commune"], data=d).fit(
          cov_type="cluster", cov_kwds={"groups": d["code_commune"]})

for v in ["lsurf", "pieces", "maison"]:
    print(f"{v:<8} HC1 {m4.bse[v]:.4f}   cluster {m4c.bse[v]:.4f}"
          f"   rapport {m4c.bse[v] / m4.bse[v]:.2f}")

**Réponse 13.** Grouper par commune suppose que les erreurs sont **corrélées à l'intérieur** d'une commune : deux ventes de la même commune subissent les mêmes chocs locaux — une nouvelle ligne de tramway, un programme neuf, un changement de PLU — que le modèle ignore.

Les écarts-types des variables qui varient surtout **entre** communes augmentent le plus ; ceux des variables qui varient **au sein** de chaque commune bougent peu. La règle pratique : groupez au niveau auquel le traitement varie, et au niveau auquel vous pensez que les chocs se produisent.


In [ ]:
ef = pd.Series({k.split("T.")[1].rstrip("]"): v
                for k, v in m4.params.items() if k.startswith("C(code_commune)")})
noms = d.drop_duplicates("code_commune").set_index("code_commune")["nom_commune"]

classement = pd.DataFrame({"effet_fixe": ef}).join(noms).sort_values("effet_fixe",
                                                                    ascending=False)
print("-- les plus cheres --")
print(classement.head(8).to_string())
print("\n-- les moins cheres --")
print(classement.tail(8).to_string())

**Réponse 14.** Les effets fixes se lisent **par rapport à la commune de référence**, celle qui a été omise : un effet fixe de 0,30 signifie 30 log-points de plus qu'elle, soit environ 35 % — à surface, pièces et type de bien identiques.

Ce classement est un vrai résultat : c'est un indice de prix communal **corrigé de la composition**. Il répond à « où le même logement coûte-t-il le plus cher ? », là où un simple prix moyen au mètre carré mélange le niveau des prix et le type de biens vendus.

**Réponse 15.** `C(code_commune) : C(trimestre)` crée une muette par commune **et par trimestre**. Cela absorbe en plus les évolutions **locales** de prix au cours de l'année — une commune qui se valorise pendant que la voisine stagne.

Le coût est double. Le nombre de paramètres est multiplié par quatre, et chaque cellule contient quatre fois moins de ventes : les effets fixes deviennent bruyants, et certaines cellules peuvent ne contenir qu'une seule vente — auquel cas elle est parfaitement expliquée et n'apporte plus rien à l'estimation. Raffiner les effets fixes n'est pas gratuit : à chaque niveau de finesse, on achète de la crédibilité avec de la précision.
